# 第 4 讲：Transformer-XL 的相对位置编码（2019）

**学习范围：**本讲先用普通 Self-Attention 解释位置编码的改进，再说明它如何配合历史记忆。位置编码、因果 Mask、历史记忆是不同的机制。

## 一、从 Shaw RPE 的距离裁剪说起

Shaw RPE 已经直接使用 Query 和 Key 的相对距离。对位置 $i,j$，它根据距离 $j-i$ 查可学习的位置向量，并将该向量用于 Attention Score（原方法也用于 Value 聚合）。

为了控制可学习参数量，Shaw 将距离限制到 $[-r,r]$。因为每一种距离都需要一个可学习表项，如果想区分更远的距离，就要增大表的行数。

例如 $r=2$ 时，距离 $2,3,7$ 都使用距离 $2$ 的位置向量：**参数量得到控制，但相对位置分支无法区分这些具体距离**。

Transformer-XL 同样根据两个 Token 的相对距离计算位置贡献，但使用**固定的 Sinusoidal 函数生成相对位置向量**，新距离不需要新增可学习的查表行。与此同时，它重新参数化原始绝对位置 Attention 中的各个匹配项。

下面先使用 3 个 Token 的普通双向 Self-Attention，不引入历史记忆或因果 Mask。

In [9]:
import math
import torch
import torch.nn as nn

# 为了先理解位置编码，使用一个没有历史缓存的 3-Token Self-Attention 示例
L = 3
pos = torch.arange(L)  # [0,1,2]，是位置编号，不是 Q/K 特征向量

# Transformer-XL 原论文使用 i-j（Query 位置 - Key 位置）
# pos[:,None]：(L,1)，行表示 Query 位置 i
# pos[None,:]：(1,L)，列表示 Key 位置 j
# 广播相减得到 (L,L)，D[i,j] = i-j
D = pos[:, None] - pos[None, :]
print('未裁剪的相对距离：\n', D)

# Shaw 示例：最大可区分距离 r=1
# clamp 只修改数值，矩阵仍为 (L,L)
# 距离 2、3、... 被归入 +1；距离 -2、-3、... 被归入 -1
r = 1
D_shaw = D.clamp(-r,r)
print('Shaw 距离裁剪后：\n', D_shaw)

未裁剪的相对距离：
 tensor([[ 0, -1, -2],
        [ 1,  0, -1],
        [ 2,  1,  0]])
Shaw 距离裁剪后：
 tensor([[ 0, -1, -1],
        [ 1,  0, -1],
        [ 1,  1,  0]])


## 二、为什么 Transformer-XL 使用正余弦位置编码？

**先区分两个问题：用什么函数表示位置，以及把什么数值传给这个函数。**

### 1. 正余弦最早用于表示绝对位置

原始 Transformer 将位置向量加到 Token Embedding 上：

$$
h_i=x_i+PE(i)
$$

其中 $i$ 是 **Token 在序列中的绝对位置编号**，例如 0、1、2。正余弦位置编码为（$k$ 是频率编号）：

$$
PE(i,2k)=\sin\left(\frac{i}{10000^{2k/d_{\mathrm{model}}}}\right),\qquad
PE(i,2k+1)=\cos\left(\frac{i}{10000^{2k/d_{\mathrm{model}}}}\right)
$$

原始 Transformer 选用正余弦，是因为它具有两个适合位置表示的性质：

- **无需查可学习表**：位置向量由公式计算，新的位置编号也可以直接代入。
- **不同频率共同表示位置与位移关系**：对于某个频率 $\omega$，设 $p(i)=[\sin(\omega i),\cos(\omega i)]$，则

$$
p(i)^\top p(j)=\cos\bigl(\omega(i-j)\bigr)
$$

这说明两组正余弦向量的内积可以与**相对距离**有关。这里仅说明 Sinusoidal 的结构特性；经过任意可学习 Q/K 投影后，完整 Attention Score 不保证只依赖 $i-j$。

### 2. Transformer-XL 将相对距离代入同一类正余弦公式

原始 Transformer 使用 $PE(i)$：输入是单个 Token 的**绝对位置** $i$。

Transformer-XL 使用 $R(\Delta)$：输入改为两个 Token 的**相对距离** $\Delta=i-j$（Query 位置减去 Key 位置，与 Shaw 教学示例中 $j-i$ 的约定相反）。

$$
R(\Delta,2k)=\sin\left(\frac{\Delta}{10000^{2k/d_{\mathrm{model}}}}\right),\qquad
R(\Delta,2k+1)=\cos\left(\frac{\Delta}{10000^{2k/d_{\mathrm{model}}}}\right)
$$

**与原始 Transformer 的区别是输入的数值：**原始 Sinusoidal 使用绝对位置 $i$，Transformer-XL 则使用相对距离 $\Delta=i-j$。例如原先计算 $PE(2)$，现在计算 $R(2)$；正余弦的计算方式不变。

### 3. 把公式变成代码：先计算一个距离

假设相对距离 $\Delta=2$、位置向量维度 $d_{\mathrm{model}}=4$。每组正余弦占两个维度，因此一共需要两组频率：

| 维度 | 计算 | 对应的角度 |
|---|---|---|
| 0 | $\sin(2/10000^0)$ | $2$ |
| 1 | $\cos(2/10000^0)$ | $2$ |
| 2 | $\sin(2/10000^{2/4})$ | $0.02$ |
| 3 | $\cos(2/10000^{2/4})$ | $0.02$ |

因此 $R(2)=[\sin(2),\cos(2),\sin(0.02),\cos(0.02)]$。

接下来把计算分成四步：**确定频率 → 得到角度 → 分别计算 sin/cos → 按维度交错排列**。代码先实现单个距离，再利用广播一次计算多个距离；多距离输入构成矩阵时仍使用同一个公式。

这正是相对 Shaw RPE 的区别：Shaw 对超过 $r$ 的距离使用边界表项，而 Sinusoidal 可以把距离 $2,3,7$ 分别代入函数计算，**不需要为这些距离新增可训练表项**。这不保证长距离建模性能一定更好。

参考：[Attention Is All You Need（2017）](https://arxiv.org/abs/1706.03762)、[Transformer-XL（2019）](https://aclanthology.org/P19-1285/)。

In [10]:
# 与上面的公式一一对应：先计算「相对距离 = 2」的四维位置向量
import torch

distance = torch.tensor(2)  # 两个 Token 的相对距离 i-j = 2
d_model = 4                # 需要输出 4 维，即两组 sin/cos

# ① 生成每组 sin/cos 对应的起始维度：0、2
# arange(0, 4, 2) 得到 [0,2]，分别对应公式中的 2k
freq_index = torch.arange(0, d_model, 2).float()  # (2,) -> [0,2]

# ② 公式中的频率因子：1 / 10000^(2k/d_model)
# 本例得到 [1,0.01]；距离乘以频率因子就得到 sin/cos 的角度
inv_freq = 1.0 / (10000 ** (freq_index / d_model))  # (2,)
angles = distance.float() * inv_freq                 # (2,) -> [2,0.02]

# ③ 同一个角度分别计算 sin 和 cos
sin_values = torch.sin(angles)  # (2,) -> [sin(2),sin(0.02)]
cos_values = torch.cos(angles)  # (2,) -> [cos(2),cos(0.02)]

# ④ stack(dim=-1) 将每个角度的 sin/cos 配成一组：(2,) + (2,) -> (2,2)
# 第 0 行：[sin(2),cos(2)]；第 1 行：[sin(0.02),cos(0.02)]
# flatten() 将两组按行连接成一个 4 维向量
R2 = torch.stack((sin_values, cos_values), dim=-1).flatten()  # (4,)
print('角度：', angles)
print('距离 2 的位置向量：', R2.round(decimals=4))

# 将以上步骤推广：distance 可以是一个数，也可以是一维或二维距离张量
def sinusoidal_relative(distance, d_model):
    # distance：整数相对距离，例如 (3,) 或 (L,L)
    # d_model：输出位置向量维度；每两维放一组 sin/cos，故要求正偶数
    if d_model <= 0 or d_model % 2 != 0:
        raise ValueError('d_model 必须是正偶数')

    # 频率索引：d_model=4 时为 [0,2]
    freq_index = torch.arange(0, d_model, 2, device=distance.device).float()
    inv_freq = 1.0 / (10000 ** (freq_index / d_model))  # (d_model/2,)

    # 在距离的最后增加一个大小为 1 的维度，留给不同频率
    # distance=(3,) -> (3,1)，乘以 (2,) 得 (3,2)：3 个距离各自有 2 个角度
    # distance=(L,L) -> (L,L,1)，乘以 (2,) 得 (L,L,2)
    angles = distance.float().unsqueeze(-1) * inv_freq

    # 对每个角度分别算 sin、cos，再让每组相邻排列
    # (3,2) -> stack 后 (3,2,2) -> flatten(-2) 后 (3,4)
    # flatten(-2) 只合并最后两个维度，保留前面的距离维度
    sin_cos_pairs = torch.stack((angles.sin(), angles.cos()), dim=-1)
    return sin_cos_pairs.flatten(-2)

# 比较 Shaw 截断距离与 Transformer-XL 按公式生成距离向量的区别
example_dist = torch.tensor([2, 3, 7])
R_far = sinusoidal_relative(example_dist, d_model=4)  # (3,4)
print('距离 2、3、7 各自的位置向量：\n', R_far.round(decimals=4))

# 用之前生成的相对距离矩阵 D（L,L）也能直接计算
R_matrix = sinusoidal_relative(D, d_model=4)  # (L,L,4)
print('整张距离矩阵对应的位置向量形状：', R_matrix.shape)

# 单个距离的手动计算，应与函数的输出一致
assert torch.allclose(R2, sinusoidal_relative(distance, d_model=4))

角度： tensor([2.0000, 0.0200])
距离 2 的位置向量： tensor([ 0.9093, -0.4161,  0.0200,  0.9998])
距离 2、3、7 各自的位置向量：
 tensor([[ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [ 0.6570,  0.7539,  0.0699,  0.9976]])
整张距离矩阵对应的位置向量形状： torch.Size([3, 3, 4])


## 三、从绝对位置 Attention 推导 Transformer-XL 的打分方式

前面已经得到了相对距离的正余弦向量 $R(i-j)$。接下来考虑：**它应该怎样进入 Attention Score？**

本节仍使用长度为 3 的普通 Self-Attention 演示公式。为了看清完整的分数矩阵，暂不使用因果 Mask；在 Transformer-XL 的自回归语言模型中会添加因果 Mask。

### 1. 先展开已经学过的绝对位置 Attention

原始 Transformer 在输入阶段将 Token 内容向量 $x_i$ 和绝对位置向量 $p_i$ 相加：

$$
h_i=x_i+p_i
$$

随后分别生成 Query 和 Key（这里采用**行向量右乘参数矩阵**的记法，与 PyTorch `nn.Linear` 的输出形式一致）：

$$
q_i^{\mathrm{abs}}=(x_i+p_i)W_Q,\qquad
k_j^{\mathrm{abs}}=(x_j+p_j)W_K
$$

把内容部分和位置部分分别记下来：

$$
\begin{aligned}
q_i^x&=x_iW_Q, & q_i^p&=p_iW_Q,\\
k_j^x&=x_jW_K, & k_j^p&=p_jW_K.
\end{aligned}
$$

因此，$q_i^{\mathrm{abs}}=q_i^x+q_i^p$，$k_j^{\mathrm{abs}}=k_j^x+k_j^p$。展开点积（暂时省略 $\sqrt{d_k}$）：

$$
\begin{aligned}
(q_i^{\mathrm{abs}})^\top k_j^{\mathrm{abs}}
&=(q_i^x+q_i^p)^\top(k_j^x+k_j^p)\\
&=\underbrace{(q_i^x)^\top k_j^x}_{\text{① 内容－内容}}
 +\underbrace{(q_i^x)^\top k_j^p}_{\text{② 内容－位置}}\\
&\quad+\underbrace{(q_i^p)^\top k_j^x}_{\text{③ 位置－内容}}
 +\underbrace{(q_i^p)^\top k_j^p}_{\text{④ 位置－位置}}.
\end{aligned}
$$

**四项是普通绝对位置 Attention 的乘法展开结果，并非 Transformer-XL 凭空增加的。** 但其中的位置部分使用 $p_i$ 和 $p_j$，依赖绝对位置编号。

### 2. 逐项改为相对位置表示

Transformer-XL 重新设计上述四项，首先定义后续始终使用的三个向量：

$$
\boxed{
q_i=x_iW_Q,\qquad
k_j=x_jW_{K,E},\qquad
r_{ij}=R(i-j)W_{K,R}
}
$$

- $q_i$：Query 的**内容向量**（对应上面的 $q_i^x$）。
- $k_j$：Key 的**内容向量**；$W_{K,E}$ 专门投影内容特征。
- $r_{ij}$：先由距离 $i-j$ 生成固定正余弦向量 $R(i-j)$，再用可学习矩阵 $W_{K,R}$ 投影到与 Query 相同的 $d_k$ 维空间。

这里 $W_{K,E}$ 与 $W_{K,R}$ **分别学习内容和位置的投影**。此前只有一套 $W_K$，现在将两种用途分开。公式采用向量右乘参数矩阵的记法；PyTorch `nn.Linear` 内部使用 `X @ weight.T`，两者只是权重矩阵存放方向不同。

对照展开的四项，可以得到下面的重新参数化方式：

| 绝对位置 Attention 中的项 | Transformer-XL 中的设计 | 为什么这样改 |
|---|---|---|
| ① $(q_i^x)^\top k_j^x$ | $q_i^\top k_j$ | 保留内容匹配 |
| ② $(q_i^x)^\top k_j^p$ | $q_i^\top r_{ij}$ | 用相对距离 $i-j$ 取代 Key 的绝对位置 |
| ③ $(q_i^p)^\top k_j^x$ | $u^\top k_j$ | 不再使用 Query 的绝对位置部分，改用共享向量 $u$ |
| ④ $(q_i^p)^\top k_j^p$ | $v^\top r_{ij}$ | 用共享向量 $v$ 和相对距离向量表示位置偏好 |

**为什么出现 $u,v$？** ③④ 原来都用到 Query 的绝对位置部分 $q_i^p$。Transformer-XL 不再使用它，而是分别设置两组可训练的 $d_k$ 维向量：$u$ 用于内容偏好，$v$ 用于相对位置偏好。它们对不同 Query 共享，但两组参数可以独立学习。

**注意：** 对照表表示新的**模型设计对应关系**，不是说 $q_i^p=u=v$，也不是原公式的代数恒等变形。仅使用①②也能建模相对位置；③④是 Transformer-XL 额外保留的全局偏好。

### 3. 得到公式，再对应代码

把四项相加并恢复缩放因子：

$$
\boxed{
s_{ij}=\frac{
\underbrace{q_i^\top k_j}_{\text{① 内容－内容}}
+\underbrace{q_i^\top r_{ij}}_{\text{② 内容－相对位置}}
+\underbrace{u^\top k_j}_{\text{③ 全局内容偏好}}
+\underbrace{v^\top r_{ij}}_{\text{④ 全局位置偏好}}
}{\sqrt{d_k}}
}
$$

其中 $r_{ij}=R(i-j)W_{K,R}$。下一格代码只计算这四项，并保持变量名与公式一致：`Q` 对应所有 $q_i$，`K` 对应所有 $k_j$，`R_K` 对应所有 $r_{ij}$，`u` 和 `v` 与公式同名。最后对 $s_{ij}$ 沿 Key 维度做 Softmax。

In [11]:
# 只演示 Transformer-XL 相对位置 Attention Score，不实现完整模型
# 本例：单头、双向 Self-Attention，B=1，L=3，d_model=d_k=4
# 与前一节公式统一：q_i、k_j、r_ij、u、v
B, L, d_model, d_k = 1, 3, 4, 4
torch.manual_seed(7)

# 1. 从同一个输入 X 生成内容 Query 和 Key
# X[b,i] 对应公式中的内容向量 x_i
X = torch.randn(B, L, d_model)            # (B,L,d_model)
W_Q = nn.Linear(d_model, d_k, bias=False) # 对应 W_Q
W_KE = nn.Linear(d_model, d_k, bias=False) # 对应 W_{K,E}
Q = W_Q(X)  # (B,L,d_k)：第 i 行就是 q_i = x_i W_Q
K = W_KE(X) # (B,L,d_k)：第 j 行就是 k_j = x_j W_{K,E}

# 2. 根据相对距离计算 r_ij = R(i-j) W_{K,R}
# D[i,j] = i-j，已在第一个代码单元格生成
R = sinusoidal_relative(D, d_model)        # (L,L,d_model)，固定正余弦向量 R(i-j)
W_KR = nn.Linear(d_model, d_k, bias=False)  # 对应 W_{K,R}，是可学习投影
R_K = W_KR(R)                              # (L,L,d_k)，第 (i,j) 格就是 r_ij

# 3. 对应公式① q_i^T k_j：内容－内容
# Q:(B,L,d_k)，K^T:(B,d_k,L) → (B,L,L)
score_1 = Q @ K.transpose(-2, -1)

# 4. 对应公式② q_i^T r_ij：内容－相对位置
# Q.unsqueeze(2)：(B,L,d_k) -> (B,L,1,d_k)，为每个 Query 增加 Key 维度
# R_K.unsqueeze(0)：(L,L,d_k) -> (1,L,L,d_k)，为位置向量增加 Batch 维度
# 相乘后沿最后的 d_k 特征维求和，就是每一对向量的点积
score_2 = (Q.unsqueeze(2) * R_K.unsqueeze(0)).sum(dim=-1)  # (B,L,L)

# 5. 对应公式③④：用可学习向量 u、v 替代 Query 的绝对位置贡献
# 这里仅演示参数值；完整模型中应把它们定义在 nn.Module.__init__ 内
u = nn.Parameter(torch.randn(d_k))  # (d_k,)，所有 Query 共享的内容偏好
v = nn.Parameter(torch.randn(d_k))  # (d_k,)，所有 Query 共享的位置偏好

# 公式③ u^T k_j：u 与每个 Key 点积
# (B,L) -> (B,1,L)，让每一行 Query 都使用这组内容偏好
score_3 = (K * u).sum(dim=-1).unsqueeze(1)  # (B,1,L)

# 公式④ v^T r_ij：v 与每一对相对位置向量点积
# (L,L) -> (1,L,L)，让不同 Batch 共享这一张位置分数矩阵
score_4 = (R_K * v).sum(dim=-1).unsqueeze(0)  # (1,L,L)

# 6. 四项相加，除以 sqrt(d_k)，再对每个 Query 的全部 Key 做 Softmax
# score_3 和 score_4 依靠广播与 (B,L,L) 对齐
scores = (score_1 + score_2 + score_3 + score_4) / math.sqrt(d_k) # (B,L,L)
weights = torch.softmax(scores, dim=-1)                           # (B,L,L)

print('四项分数形状：', score_1.shape, score_2.shape, score_3.shape, score_4.shape)
print('Attention Score 形状：', scores.shape)
print('第 2 个 Query 对三个 Key 的权重：', weights[0,1].round(decimals=4))
print('每行权重之和：', weights.sum(dim=-1))

四项分数形状： torch.Size([1, 3, 3]) torch.Size([1, 3, 3]) torch.Size([1, 1, 3]) torch.Size([1, 3, 3])
Attention Score 形状： torch.Size([1, 3, 3])
第 2 个 Query 对三个 Key 的权重： tensor([0.3082, 0.4037, 0.2881], grad_fn=<RoundBackward1>)
每行权重之和： tensor([[1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)



## 四、Transformer-XL 如何利用历史记忆？

### 1. 为什么需要保存历史信息？

普通 Transformer 处理较长文本时，可以将文本分成多个片段。

例如：

```text
第一段：苹果  很  甜      位置 [0,1,2]
第二段：而且  新鲜        位置 [3,4]
```

如果分别处理两个片段，第二段中的「新鲜」就无法通过 Self-Attention 直接关注第一段的「苹果」「很」「甜」。

因此，Transformer-XL 引入了**历史记忆（Segment-Level Recurrence）**：

处理完第一段后，保存它的 Hidden States（隐藏状态）。处理第二段时，将这些历史状态作为额外的上下文，让当前 Token 也能关注第一段的信息。

历史状态在复用时会停止梯度传播，避免反向传播无限延伸到之前的片段。

### 2. 保存历史后，Self-Attention 有什么变化？

普通 Self-Attention 中，Q、K、V 都来自当前序列。

引入历史记忆后：

- **Q**：仍然只来自当前段「而且、新鲜」，长度为 2。
- **K、V**：来自历史段和当前段，共有 5 个 Token。

因此，Attention Score 的形状不再是 $(2,2)$，而是：

$$
(L_q,L_k)=(2,5)
$$

对应的 Attention 矩阵：

| Query \ Key | 苹果(0) | 很(1) | 甜(2) | 而且(3) | 新鲜(4) |
|---|---|---|---|---|---|
| 而且(3) | ✓ | ✓ | ✓ | ✓ | × |
| 新鲜(4) | ✓ | ✓ | ✓ | ✓ | ✓ |

其中 ✓ 表示允许关注，× 表示被因果 Mask 屏蔽。

虽然 Query 来自当前段，Key/Value 包含历史段，但它们仍属于同一文本序列，因此这里仍然是**因果 Self-Attention**。

### 3. 相对位置编码如何处理历史 Token？

Transformer-XL 使用相对距离：

$$
D_{ij}=i-j
$$

这里 $i$ 是 Query 的位置，$j$ 是 Key 的位置。

当前 Query 的位置是 $[3,4]$，表示的是位置3跟位置4，所有 Key 的位置是 $[0,1,2,3,4]$，把前面那段也保存下来了。

因此：

$$
D=
\begin{bmatrix}
3&2&1&0&-1\\
4&3&2&1&0
\end{bmatrix}
$$

例如，「新鲜」位于位置 4，「苹果」位于位置 0，因此相对距离为 $4-0=4$。

接下来仍然使用前面学过的 Sinusoidal 函数：

$$
R(i-j)
$$

为每一个相对距离生成位置向量，再代入 Transformer-XL 的 Attention Score 公式。

**这与普通 Self-Attention 的相对位置计算没有本质区别，只是现在需要计算当前 Token 与更多历史 Token 之间的距离。**

### 4. 历史记忆与相对位置编码的关系

Transformer-XL 将两种机制结合起来：

- **历史记忆**：保存上一段的隐藏状态，让当前段能够关注更早的 Token。
- **Sinusoidal 相对位置编码**：根据 Token 之间的相对距离生成位置向量，不需要为每个新的绝对位置学习独立的 Embedding。

两种机制分别解决上下文复用和位置表示问题，也可以独立使用。


In [ ]:

# 1. 模拟带历史记忆的因果 Self-Attention
# 完整序列：苹果(0) 很(1) 甜(2) 而且(3) 新鲜(4)
# 历史段：苹果、很、甜，共 3 个 Token
# 当前段：而且、新鲜，共 2 个 Token

M = 3          # 历史段的长度
L = 2          # 当前段的长度
start_pos = 3  # 当前段第一个 Token 在完整序列中的位置是 3

# 2. 计算当前段 Query 的位置
# torch.arange(L) = [0,1]，表示当前段内部的两个位置
# 加上 start_pos=3，得到它们在完整序列中的位置 [3,4]
# 当前段只有两个 Token，因此只需要两个 Query
query_pos = start_pos + torch.arange(L)  # (L,) -> [3,4]

# 3. 计算历史段 + 当前段所有 Key 的位置
# torch.arange(-M,L) = [-3,-2,-1,0,1]
# 其中 -3,-2,-1 表示相对于当前段起点，历史 Token 的位置偏移
# 0,1 表示当前段两个 Token 的位置偏移
# 加上 start_pos=3，得到完整序列中的位置 [0,1,2,3,4]
# Key 包含历史段和当前段，因此长度为 M+L=5
key_pos = start_pos + torch.arange(-M,L)  # (M+L,) -> [0,1,2,3,4]

# M缓存的如果只是前面两个，那就得是[1,2,3,4]

# 4. 计算 Query 与 Key 的相对距离 D[i,j] = Query位置 - Key位置
# query_pos[:,None]：(2,) -> (2,1)，每一行对应一个 Query
# key_pos[None,:]：(5,) -> (1,5)，每一列对应一个 Key
# 利用广播相减，得到 (2,5) 的相对距离矩阵
# 第一行：[3-0,3-1,3-2,3-3,3-4] = [3,2,1,0,-1]
# 第二行：[4-0,4-1,4-2,4-3,4-4] = [4,3,2,1,0]
D_memory = query_pos[:,None] - key_pos[None,:]  # (L,M+L)
print('带历史时的相对距离矩阵：\n', D_memory)

# 5. 构造因果 Mask，防止当前 Query 关注未来的 Key
# 只有 Key位置 <= Query位置，才允许该 Query 关注这个 Key
# 利用广播逐个比较，得到 (2,5) 的布尔矩阵
# 第一行：[True,True,True,True,False]，位置 3 不能关注位置 4
# 第二行：[True,True,True,True,True]，位置 4 可以关注所有已有 Token
# True 表示允许关注，False 表示需要屏蔽
causal_mask = key_pos[None,:] <= query_pos[:,None]  # (L,M+L)
print('因果 Mask：\n', causal_mask)

# 6. 将相对距离转换成 Sinusoidal 位置向量
# D_memory：(L,M+L)，其中每个元素表示一对 Query-Key 的相对距离
# sinusoidal_relative 会将每个整数距离转换成 d_model 维位置向量
# 因此输出形状为 (L,M+L,d_model)，即 (2,5,4)
# 相对位置编码的计算方式没有变化，只是现在 Key 包含历史 Token
R_memory = sinusoidal_relative(D_memory, d_model=4)  # (L,M+L,d_model)
print('相对位置向量形状：', R_memory.shape)


带历史时的相对距离矩阵：
 tensor([[ 3,  2,  1,  0, -1],
        [ 4,  3,  2,  1,  0]])
因果 Mask：
 tensor([[ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])
相对位置向量形状： torch.Size([2, 5, 4])
